# 04 - Effective Receptive Field (ERF) Analysis

Gradient-based ERF (Luo et al. 2016) for V1/V2/V3 at P3/P4/P5.
Produces heatmaps and quantitative metrics.

In [ ]:
# Zelle 0 - Imports
import csv
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import seaborn as sns

In [ ]:
# Zelle 1 - Setup + Drive
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO = '/content/ba-mamba-neck'
os.chdir(REPO)
sys.path.insert(0, REPO)

In [ ]:
# Zelle 2 - Run ERF analysis (writes heatmaps + CSV)
!PYTHONPATH=. python eval/erf.py \
    --data-root /content/visdrone \
    --ckpt-dir /content/drive/MyDrive/ba \
    --results results \
    --n-images 500

In [ ]:
# Zelle 3 - 3x3 ERF comparison panel + quantitative table
from eval.plot_style import apply_style, TEXTWIDTH
from eval.constants import NECKS, NECK_LABELS
apply_style()

LEVELS = ['P3', 'P4', 'P5']
HEATMAP_DIR = Path('results/erf_heatmaps')
FIG_DIR = Path('results/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- 3x3 panel ---
fig, axes = plt.subplots(3, 3, figsize=(TEXTWIDTH, TEXTWIDTH))
for j, neck in enumerate(NECKS):
    for i, level in enumerate(LEVELS):
        img_path = HEATMAP_DIR / f'{neck}_{level}.png'
        if img_path.exists():
            img = mpimg.imread(str(img_path))
            axes[i, j].imshow(img)
        axes[i, j].set_title(f'{NECK_LABELS[neck]} - {level}', fontsize=10)
        axes[i, j].axis('off')
fig.savefig(FIG_DIR / 'erf_comparison.pdf')
plt.show()
print(f'Saved: {FIG_DIR / "erf_comparison.pdf"}')

# --- P4-only panel ---
fig, axes = plt.subplots(1, 3, figsize=(TEXTWIDTH, TEXTWIDTH * 0.35))
for j, neck in enumerate(NECKS):
    img_path = HEATMAP_DIR / f'{neck}_P4.png'
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        axes[j].imshow(img)
    axes[j].set_title(NECK_LABELS[neck], fontsize=10)
    axes[j].axis('off')
fig.savefig(FIG_DIR / 'erf_comparison_P4_only.pdf')
plt.show()
print(f'Saved: {FIG_DIR / "erf_comparison_P4_only.pdf"}')

# --- Quantitative table ---
quant_path = Path('results/erf_quantitative.csv')
if quant_path.exists():
    with quant_path.open() as f:
        rows = list(csv.DictReader(f))
    print('\nERF Quantitative Metrics:')
    print(f'{"Neck":<8} {"Level":<5} {"Area@10%":>10} {"Area@50%":>10} '
          f'{"Gini":>8} {"Entropy":>10} {"MaxDist":>10}')
    print('-' * 65)
    for r in rows:
        print(f'{r["neck"]:<8} {r["level"]:<5} {r["area_10"]:>10} '
              f'{r["area_50"]:>10} {r["gini"]:>8} {r["entropy"]:>10} '
              f'{r["max_distance"]:>10}')
else:
    print('erf_quantitative.csv not found')